# Roteiro: Montando a Stack de Dados do Zero

Guia passo a passo para construir uma stack completa de processamento de dados com Docker.

| Parte | O que monta | Resultado |
|-------|------------|----------|
| **1** | PySpark + Jupyter | Cluster Spark com notebooks interativos |
| **2** | + MinIO | Object storage S3-compatible para Data Lake |
| **3** | + Dremio | Query engine SQL sobre os dados do lake |

Cada parte e independente — voce pode parar em qualquer fase e ja ter uma stack funcional.

---

# PARTE 1 — Spark + Jupyter

### O que vamos montar

```
          +-------------------+
          |  Jupyter (8888)   |
          |  (Spark Driver)   |
          +--------+----------+
                   |
          +--------v----------+
          | Spark Master (8090)|
          +--------+----------+
                   |
        +----------+----------+
        |          |          |
   +----v---+ +----v---+ +----v---+
   |Worker 1| |Worker 2| |Worker 3|
   |(8081)  | |(8082)  | |(8083)  |
   +--------+ +--------+ +--------+

   +-------------------+
   | Spark History     |
   | (18080)           |
   +-------------------+
```

**6 containers:** 1 master, 3 workers, 1 history server, 1 jupyter

### Conceitos

- **Spark Master** — coordena o cluster, distribui tarefas
- **Spark Workers** — executam o processamento de dados
- **Jupyter** — IDE interativa que atua como driver Spark (envia comandos ao cluster)
- **History Server** — armazena historico de jobs para analise posterior
- **Modo standalone** — Spark gerencia seus proprios recursos (sem YARN/Mesos/K8s)

### 1.1 — Estrutura de arquivos

Crie a seguinte estrutura de pastas:

```
stack-prev/
├── docker-compose.yml
├── Dockerfile.spark
├── requirements.txt
└── config/
    └── spark/
        ├── spark-defaults.conf
        ├── log4j2.properties
        ├── jupyter-entrypoint.sh
        └── jars/
            ├── delta-spark_2.12-3.2.0.jar
            └── delta-storage-3.2.0.jar
```

```bash
mkdir -p stack-prev/config/spark/jars
mkdir -p stack-prev/data/spark-events
mkdir -p stack-prev/work
cd stack-prev
```

### 1.2 — Dockerfile.spark

Imagem customizada baseada no `bitnami/spark:3.5.5`. Instala Python, pip, libs e JARs extras.

```dockerfile
FROM bitnami/spark:3.5.5

USER root

RUN apt-get update && \
    apt-get install -y --no-install-recommends python3 python3-pip curl && \
    apt-get clean && \
    rm -rf /var/lib/apt/lists/*

RUN mkdir -p /opt/bitnami/spark/logs/events && \
    chmod 777 /opt/bitnami/spark/logs/events && \
    chmod 777 /opt/bitnami/spark/logs

COPY requirements.txt /tmp/requirements.txt
RUN pip3 install --no-cache-dir -r /tmp/requirements.txt && \
    rm /tmp/requirements.txt

COPY config/spark/spark-defaults.conf /opt/bitnami/spark/conf/
COPY config/spark/log4j2.properties /opt/bitnami/spark/conf/
COPY config/spark/jars /opt/bitnami/spark/jars

EXPOSE 8080 7077 18080

USER 1001
```

**Por que bitnami?** Imagem oficial otimizada, com scripts de inicializacao prontos para modo master/worker via variavel de ambiente.

### 1.3 — requirements.txt

```
pyspark==3.5.1
pandas==2.2.2
pyarrow==16.1.0
sparkmeasure==0.24.0
deltalake==0.17.4
delta-spark==3.2.0
jupyter==1.0.0
notebook==6.5.7
psycopg2-binary==2.9.9
sqlalchemy==2.0.20
```

### 1.4 — config/spark/spark-defaults.conf (versao basica)

Nesta fase, so precisamos do basico: master, event log e limites de recursos.

```properties
spark.master                     spark://spark-master:7077
spark.eventLog.enabled           true
spark.eventLog.dir               file:/opt/bitnami/spark/logs/events
spark.history.fs.logDirectory    file:/opt/bitnami/spark/logs/events
spark.history.provider           org.apache.spark.deploy.history.FsHistoryProvider

# Limites por aplicacao (protege o cluster)
spark.driver.memory              512m
spark.cores.max                  2
spark.executor.memory            1g
```

**O que cada config faz:**
- `spark.master` — todo SparkSession criado no Jupyter ja conecta ao cluster automaticamente
- `spark.eventLog.*` — habilita historico de jobs para o History Server
- `spark.driver.memory` — limita o driver (Jupyter) a 512 MB
- `spark.cores.max` — cada app usa no maximo 2 dos 6 cores disponiveis
- `spark.executor.memory` — cada executor limitado a 1 GB

### 1.5 — config/spark/log4j2.properties

Reduz o nivel de log para `warn` (evita excesso de mensagens no Jupyter).

```properties
status = error
name = PropertiesConfig

property.filename = /opt/bitnami/spark/logs/spark.log

appenders = console, file

appender.console.type = Console
appender.console.name = console
appender.console.layout.type = PatternLayout
appender.console.layout.pattern = %d{ISO8601} [%t] %-5p %c %x - %m%n

appender.file.type = File
appender.file.name = file
appender.file.fileName = ${filename}
appender.file.layout.type = PatternLayout
appender.file.layout.pattern = %d{ISO8601} [%t] %-5p %c %x - %m%n

rootLogger.level = warn
rootLogger.appenderRefs = console, file
rootLogger.appenderRef.console.ref = console
rootLogger.appenderRef.file.ref = file

logger.py4j.name = py4j
logger.py4j.level = warn
logger.py4j.additivity = false
logger.py4j.appenderRefs = console, file
logger.py4j.appenderRef.console.ref = console
logger.py4j.appenderRef.file.ref = file
```

### 1.6 — config/spark/jupyter-entrypoint.sh

Script que configura o `spark.driver.host` dinamicamente para cada container Jupyter.

```bash
#!/bin/bash
# Entrypoint para containers Jupyter
# Cria config Spark personalizada com driver.host do container

cp /opt/bitnami/spark/conf/spark-defaults.conf /tmp/spark-defaults.conf
echo "spark.driver.host                ${SPARK_DRIVER_HOST:-jupyter-1}" >> /tmp/spark-defaults.conf
echo "spark.driver.bindAddress         0.0.0.0" >> /tmp/spark-defaults.conf

export SPARK_CONF_DIR=/tmp

exec jupyter notebook \
    --ip=0.0.0.0 \
    --port=8888 \
    --no-browser \
    --allow-root \
    --NotebookApp.token="${JUPYTER_TOKEN:-spark123}" \
    --NotebookApp.password= \
    --notebook-dir=/opt/bitnami/spark/work
```

```bash
# Tornar executavel
chmod +x config/spark/jupyter-entrypoint.sh
```

**Por que esse script?**
O `spark.driver.host` precisa ser o nome do container (ex: `jupyter-1`). Como o spark-defaults.conf e compartilhado entre todos os containers, o entrypoint copia a config base para `/tmp`, adiciona o driver.host correto e inicia o Jupyter apontando para essa config.

### 1.7 — docker-compose.yml (Parte 1: so Spark + Jupyter)

```yaml
services:
  spark-master:
    build:
      context: .
      dockerfile: Dockerfile.spark
    container_name: spark-master
    environment:
      - SPARK_MODE=master
      - SPARK_MASTER_HOST=spark-master
      - SPARK_MASTER_PORT=7077
      - SPARK_MASTER_WEBUI_PORT=8080
    deploy:
      resources:
        limits:
          memory: 1g
          cpus: '1.0'
    ports:
      - "8090:8080"   # Spark Master UI
      - "7077:7077"   # Spark Master port
    volumes:
      - ./config/spark/spark-defaults.conf:/opt/bitnami/spark/conf/spark-defaults.conf
      - ./config/spark/log4j2.properties:/opt/bitnami/spark/conf/log4j2.properties
      - ./data/spark-events:/opt/bitnami/spark/logs/events
    networks:
      - spark-network

  spark-worker-1:
    build:
      context: .
      dockerfile: Dockerfile.spark
    container_name: spark-worker-1
    environment:
      - SPARK_MODE=worker
      - SPARK_MASTER_URL=spark://spark-master:7077
      - SPARK_WORKER_MEMORY=2g
      - SPARK_WORKER_CORES=2
      - SPARK_WORKER_WEBUI_PORT=8081
      - SPARK_WORKER_DIR=/tmp/spark-worker
    deploy:
      resources:
        limits:
          memory: 2.5g
          cpus: '2.0'
    ports:
      - "8081:8081"
    volumes:
      - ./config/spark/spark-defaults.conf:/opt/bitnami/spark/conf/spark-defaults.conf
      - ./config/spark/log4j2.properties:/opt/bitnami/spark/conf/log4j2.properties
      - ./data/spark-events:/opt/bitnami/spark/logs/events
    depends_on:
      - spark-master
    networks:
      - spark-network

  # spark-worker-2 e spark-worker-3: identicos ao worker-1
  # Mude apenas: container_name, porta host (8082, 8083)

  spark-history:
    build:
      context: .
      dockerfile: Dockerfile.spark
    container_name: spark-history
    command:
      - /opt/bitnami/spark/bin/spark-class
      - org.apache.spark.deploy.history.HistoryServer
    deploy:
      resources:
        limits:
          memory: 512m
          cpus: '0.5'
    ports:
      - "18080:18080"
    volumes:
      - ./config/spark/spark-defaults.conf:/opt/bitnami/spark/conf/spark-defaults.conf
      - ./config/spark/log4j2.properties:/opt/bitnami/spark/conf/log4j2.properties
      - ./data/spark-events:/opt/bitnami/spark/logs/events
    depends_on:
      - spark-master
    networks:
      - spark-network

  jupyter-1:
    build:
      context: .
      dockerfile: Dockerfile.spark
    container_name: jupyter-1
    user: root
    command:
      - /bin/bash
      - /opt/bitnami/spark/conf/jupyter-entrypoint.sh
    environment:
      - SPARK_MASTER_URL=spark://spark-master:7077
      - PYSPARK_PYTHON=python3
      - PYSPARK_DRIVER_PYTHON=python3
      - SPARK_DRIVER_HOST=jupyter-1
      - JUPYTER_TOKEN=${JUPYTER_TOKEN:-spark123}
    deploy:
      resources:
        limits:
          memory: 2g
          cpus: '1.0'
    ports:
      - "8888:8888"
    volumes:
      - ./config/spark/spark-defaults.conf:/opt/bitnami/spark/conf/spark-defaults.conf
      - ./config/spark/log4j2.properties:/opt/bitnami/spark/conf/log4j2.properties
      - ./config/spark/jupyter-entrypoint.sh:/opt/bitnami/spark/conf/jupyter-entrypoint.sh
      - ./data/spark-events:/opt/bitnami/spark/logs/events
      - ./work:/opt/bitnami/spark/work
    depends_on:
      - spark-master
    networks:
      - spark-network

networks:
  spark-network:
    driver: bridge
```

### 1.8 — Subir e validar

```bash
# Criar pastas de dados
mkdir -p data/spark-events work

# Build e subir
docker compose up -d --build

# Verificar (6 containers Up)
docker compose ps
```

**URLs para validar:**

| Servico | URL | O que ver |
|---|---|---|
| Spark Master | http://localhost:8090 | 3 workers registrados |
| Worker 1 | http://localhost:8081 | 2 cores, 2 GB |
| Worker 2 | http://localhost:8082 | 2 cores, 2 GB |
| Worker 3 | http://localhost:8083 | 2 cores, 2 GB |
| History Server | http://localhost:18080 | Lista de apps executadas |
| Jupyter | http://localhost:8888/?token=spark123 | Interface de notebooks |

### 1.9 — Testar o cluster

Abra o Jupyter (http://localhost:8888/?token=spark123), crie um notebook e execute:

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("teste-cluster") \
    .getOrCreate()

# Verificar conexao
print(f"Spark {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"Driver: {spark.conf.get('spark.driver.host')}")

# Teste basico — criar DataFrame e processar
dados = [(i, f"nome_{i}", i * 10) for i in range(1000)]
df = spark.createDataFrame(dados, ["id", "nome", "valor"])

print(f"Registros: {df.count()}")
print(f"Particoes: {df.rdd.getNumPartitions()}")
df.groupBy().avg("valor").show()

spark.stop()
print("Cluster funcionando!")

### 1.10 — Troubleshooting Parte 1

| Problema | Causa | Solucao |
|---|---|---|
| Workers nao aparecem no Master UI | Demora para registrar | Aguarde 30s e atualize |
| `Initial job has not accepted any resources` | Driver e workers nao se comunicam | Verifique se `spark.driver.host` esta correto no `/tmp/spark-defaults.conf` |
| Jupyter 403 Forbidden | Token nao informado | Acesse com `?token=spark123` na URL |
| Build demora muito | Primeira vez baixa imagens e pip install | Normal, ~5-10 min. Builds seguintes usam cache |

**A Parte 1 esta completa.** Voce tem um cluster Spark funcional com Jupyter para processar dados em memoria.

---

# PARTE 2 — Adicionar MinIO (Object Storage S3)

### O que muda

```
          +-------------------+
          |  Jupyter (8888)   |-----+
          |  (Spark Driver)   |     |
          +--------+----------+     |  S3A protocol
                   |                |  (leitura/escrita)
          +--------v----------+     |
          | Spark Master (8090)|    |
          +--------+----------+     |
                   |                |
        +----------+----------+     |
        |          |          |     |
   +----v---+ +----v---+ +----v---+ |
   |Worker 1| |Worker 2| |Worker 3| |
   +----+---+ +----+---+ +----+---+ |
        |          |          |      |
        +----------+----------+------+
                   |
   +-------+ +-------+ +-------+ +-------+
   |MinIO 1| |MinIO 2| |MinIO 3| |MinIO 4|
   |(9000) | |       | |       | |       |
   |(9001) | |       | |       | |       |
   +-------+ +-------+ +-------+ +-------+
```

**O que e o MinIO?** Object storage S3-compatible que roda localmente. Funciona como um Amazon S3, mas no seu Docker.

**Por que 4 nodes?** Modo distribuido com erasure coding — redundancia de dados. Mesmo que 1 node caia, os dados continuam acessiveis.

**O que muda na stack:**
1. Adicionar 4 services MinIO no `docker-compose.yml`
2. Adicionar configs S3A no `spark-defaults.conf`
3. Os JARs `hadoop-aws` e `aws-java-sdk` ja vem no bitnami/spark

### 2.1 — Adicionar MinIO ao docker-compose.yml

Adicione estes services **apos** os services do Spark:

```yaml
  minio1:
    image: minio/minio
    container_name: minio1
    environment:
      - MINIO_ROOT_USER=minioadmin
      - MINIO_ROOT_PASSWORD=minioadmin
    command: server http://minio{1...4}/data --console-address ":9001"
    ports:
      - "9000:9000"   # API S3
      - "9001:9001"   # Console Web
    volumes:
      - ./data/minio1:/data
    networks:
      - spark-network

  minio2:
    image: minio/minio
    container_name: minio2
    environment:
      - MINIO_ROOT_USER=minioadmin
      - MINIO_ROOT_PASSWORD=minioadmin
    command: server http://minio{1...4}/data --console-address ":9001"
    volumes:
      - ./data/minio2:/data
    networks:
      - spark-network

  minio3:
    image: minio/minio
    container_name: minio3
    environment:
      - MINIO_ROOT_USER=minioadmin
      - MINIO_ROOT_PASSWORD=minioadmin
    command: server http://minio{1...4}/data --console-address ":9001"
    volumes:
      - ./data/minio3:/data
    networks:
      - spark-network

  minio4:
    image: minio/minio
    container_name: minio4
    environment:
      - MINIO_ROOT_USER=minioadmin
      - MINIO_ROOT_PASSWORD=minioadmin
    command: server http://minio{1...4}/data --console-address ":9001"
    volumes:
      - ./data/minio4:/data
    networks:
      - spark-network
```

**Observacoes:**
- So o `minio1` expoe as portas 9000/9001 (os outros sao internos)
- O comando `http://minio{1...4}/data` configura o cluster distribuido
- Credenciais padrao: `minioadmin` / `minioadmin`

### 2.2 — Atualizar spark-defaults.conf

Adicione estas linhas ao final do `spark-defaults.conf`:

```properties
# MinIO S3A
spark.hadoop.fs.s3a.endpoint              http://minio1:9000
spark.hadoop.fs.s3a.access.key            minioadmin
spark.hadoop.fs.s3a.secret.key            minioadmin
spark.hadoop.fs.s3a.path.style.access     true
spark.hadoop.fs.s3a.impl                  org.apache.hadoop.fs.s3a.S3AFileSystem
spark.hadoop.fs.s3a.connection.ssl.enabled false
```

**O que cada config faz:**
- `endpoint` — URL do MinIO dentro da rede Docker (container `minio1`, porta 9000)
- `access.key` / `secret.key` — credenciais (iguais ao `MINIO_ROOT_USER/PASSWORD`)
- `path.style.access` — obrigatorio para MinIO (nao usa virtual-hosted style)
- `impl` — implementacao do filesystem S3A do Hadoop
- `connection.ssl.enabled` — desabilitado (sem HTTPS dentro do Docker)

**Com essas configs, qualquer SparkSession criada no Jupyter ja acessa o MinIO automaticamente via `s3a://`**

### 2.3 — Subir e validar

```bash
# Criar pastas do MinIO
mkdir -p data/minio1 data/minio2 data/minio3 data/minio4

# Rebuild (spark-defaults.conf mudou, precisa rebuild da imagem)
docker compose up -d --build

# Verificar (10 containers Up)
docker compose ps
```

**Acessar MinIO Console:** http://localhost:9001
- Login: `minioadmin` / `minioadmin`

### 2.4 — Criar buckets no MinIO

No MinIO Console (http://localhost:9001):

1. Menu lateral > **Buckets** > **Create Bucket**
2. Criar os seguintes buckets (um por vez):

| Bucket | Funcao |
|--------|--------|
| `landing` | Dados brutos (JSON de origem) |
| `bronze` | Dados ingeridos (Parquet/Delta cru + metadados) |
| `prata` | Dados limpos e padronizados |
| `ouro` | Agregacoes de negocio prontas para consumo |

3. Fazer upload de arquivos JSON no bucket `landing`:
   - Object Browser > **landing** > Upload > selecionar arquivos `.json`

### 2.5 — Testar leitura/escrita no MinIO

No Jupyter, crie um notebook e execute:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit

spark = SparkSession.builder \
    .appName("teste-minio") \
    .getOrCreate()

# Ler JSON do bucket landing
df = spark.read.json("s3a://landing/*.json")
print(f"Registros do landing: {df.count()}")
df.show(5)

# Gravar como Parquet no bucket bronze
df.write.mode("overwrite").parquet("s3a://bronze/device/parquet")
print("Parquet gravado no bronze!")

# Ler de volta
df_parquet = spark.read.parquet("s3a://bronze/device/parquet")
print(f"Lido do bronze: {df_parquet.count()} registros")

spark.stop()

### 2.6 — Usar Delta Lake (opcional nesta fase)

Para usar Delta Lake, adicione 3 configs extras na SparkSession:

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("teste-delta") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore") \
    .getOrCreate()

# Ler do landing e gravar como Delta no bronze
df = spark.read.json("s3a://landing/*.json")
df.write.format("delta").mode("overwrite").save("s3a://bronze/device/delta")
print("Delta gravado!")

# Ler Delta
df_delta = spark.read.format("delta").load("s3a://bronze/device/delta")
print(f"Delta: {df_delta.count()} registros")

spark.stop()

### 2.7 — Troubleshooting Parte 2

| Problema | Causa | Solucao |
|---|---|---|
| MinIO nao inicia | Dados corrompidos | `rm -rf data/minio* && mkdir -p data/minio{1..4}` e recriar |
| `No FileSystem for scheme: s3a` | JAR hadoop-aws ausente | Ja vem no bitnami. Verifique se o spark-defaults.conf esta montado |
| `403 Access Denied` no S3A | Credenciais erradas | Verifique access.key/secret.key no spark-defaults.conf |
| `Connection refused` no endpoint | MinIO nao subiu | `docker compose logs minio1 --tail 20` |

**A Parte 2 esta completa.** Agora voce tem um Data Lake com storage persistente.

---

# PARTE 3 — Adicionar Dremio (Query Engine SQL)

### O que muda

```
   +--------+    +--------+
   |Jupyter1|    |Jupyter2|          +---------+
   +---+----+    +----+---+          | Dremio  |
       |              |              | (9047)  |
       +------+-------+              +----+----+
              |                           |
       +------v-------+                   |
       | Spark Master |                   |
       +------+-------+                   |
              |                           |
   +----------+----------+                |
   |          |          |                |
   v          v          v                |
 Worker1  Worker2  Worker3               |
   |          |          |                |
   +----------+----------+----------------+
              |
   +------+------+------+------+
   |MinIO1|MinIO2|MinIO3|MinIO4|
   +------+------+------+------+
```

**O que e o Dremio?** Motor SQL que conecta diretamente no MinIO e permite consultar dados Parquet/Delta como tabelas SQL. Substitui a necessidade de codigo Python para consultas simples.

**Para que serve?**
- Consultas SQL ad-hoc sobre o Data Lake
- Alimentar dashboards (Metabase, Grafana, PowerBI via ODBC)
- Virtualizar dados sem mover/copiar

### 3.1 — Adicionar Dremio ao docker-compose.yml

Adicione este service ao `docker-compose.yml`:

```yaml
  dremio:
    image: dremio/dremio-oss
    container_name: dremio
    environment:
      - DREMIO_MAX_HEAP_MEMORY_SIZE_MB=4096
      - DREMIO_MAX_DIRECT_MEMORY_SIZE_MB=2048
    deploy:
      resources:
        limits:
          memory: 8g
    ports:
      - "9047:9047"    # Web UI
      - "31010:31010"  # ODBC/JDBC
      - "32010:32010"  # Arrow Flight
      - "45678:45678"  # Inter-node
    volumes:
      - ./data/dremio:/opt/dremio/data
      - ./data/dremio-spill:/opt/dremio/spill
    networks:
      - spark-network
```

### 3.2 — Subir e validar

```bash
# Criar pastas do Dremio e dar permissao
mkdir -p data/dremio data/dremio-spill
chmod -R 777 data/dremio data/dremio-spill

# Subir (nao precisa rebuild — Dremio usa imagem oficial)
docker compose up -d

# Verificar (13 containers Up)
docker compose ps
```

**IMPORTANTE:** O `chmod 777` e obrigatorio. O Dremio roda com um usuario interno (nao root). Sem permissao, falha com `path /opt/dremio/data is not writable`.

**Acessar Dremio:** http://localhost:9047

### 3.3 — Primeiro acesso ao Dremio

1. Abra http://localhost:9047
2. Crie a conta de administrador:
   - **Username:** `admin`
   - **Password:** (minimo 8 caracteres)
3. Clique **Save** / **Next**

### 3.4 — Conectar Dremio ao MinIO

1. Clique em **Add Source** (botao `+` no painel esquerdo)
2. Selecione **Amazon S3**

**Aba General:**

| Campo | Valor |
|---|---|
| Name | `dremio_minio` |
| AWS Access Key | `minioadmin` |
| AWS Access Secret | `minioadmin` |
| Encrypt connection | **Desmarcar** |

Na secao **Public Buckets**, adicione: `landing`, `bronze`, `prata`, `ouro`

**Aba Advanced Options:**

Marcar:
- Enable asynchronous access when possible
- Enable compatibility mode
- Enable file status check

**Connection Properties** (clique Add property):

| Property Name | Property Value |
|---|---|
| `fs.s3a.endpoint` | `minio1:9000` |
| `fs.s3a.path.style.access` | `true` |
| `dremio.s3.compact` | `true` |

**Cache Options:**
- Enable local caching: **Marcado**
- Max percent: `50`

Clique **Save**.

### 3.5 — Promover pastas para tabelas

No painel esquerdo, navegue: `dremio_minio` > `bronze` > `device` > `parquet`

1. Clique no icone de **promover** (icone de tabela)
2. Em Format, selecione **Parquet**
3. Clique **Save**

Para Delta Lake:
1. Navegue ate a pasta `delta`
2. Promova com Format: **Delta Lake**

Agora consulte via SQL no Dremio:

```sql
-- Contar registros
SELECT COUNT(*) FROM dremio_minio.bronze.device.parquet;

-- Agrupar por fabricante
SELECT manufacturer, COUNT(*) as total
FROM dremio_minio.bronze.device.delta
GROUP BY manufacturer
ORDER BY total DESC;

-- Filtrar
SELECT * FROM dremio_minio.bronze.device.delta
WHERE platform = 'Android'
LIMIT 100;
```

### 3.6 — Integracoes

O Dremio expoe os dados via:

| Protocolo | Porta | Para que |
|---|---|---|
| ODBC/JDBC | 31010 | PowerBI, Metabase, DBeaver |
| Arrow Flight | 32010 | Python (pyarrow), alta performance |
| REST API | 9047 | Aplicacoes web |

**Exemplo: conectar Metabase ao Dremio**
- Tipo: PostgreSQL
- Host: `dremio` (ou `localhost` se fora do Docker)
- Porta: `31010`
- Database: `dremio_minio`
- Username/Password: as credenciais do admin Dremio

### 3.7 — Troubleshooting Parte 3

| Problema | Causa | Solucao |
|---|---|---|
| `path /opt/dremio/data is not writable` | Sem permissao | `chmod -R 777 data/dremio data/dremio-spill && docker compose restart dremio` |
| Dremio nao conecta no MinIO | Endpoint errado | Usar `minio1:9000` (nome do container, nao `localhost`) |
| Source nao mostra buckets | Falta compatibility mode | Marcar **Enable compatibility mode** nas Advanced Options |
| Dremio demora para iniciar | Normal, JVM pesada | Aguarde ~1 minuto apos o container subir |

---

# Resumo Final

### Stack completa: 13 containers

| Fase | Services | Portas |
|------|----------|--------|
| **Parte 1** | spark-master, 3 workers, history, jupyter-1 | 7077, 8081-8083, 8090, 8888, 18080 |
| **Parte 2** | + minio1, minio2, minio3, minio4 | + 9000, 9001 |
| **Parte 3** | + dremio | + 9047, 31010, 32010 |

### Limites de recursos

| Servico | Memoria | CPUs |
|---|---|---|
| spark-master | 1 GB | 1.0 |
| spark-worker (x3) | 2.5 GB | 2.0 |
| spark-history | 512 MB | 0.5 |
| jupyter (x2) | 2 GB | 1.0 |
| dremio | 8 GB | - |
| **Total** | **~24 GB** | - |

### Comandos essenciais

```bash
# Subir tudo
docker compose up -d --build

# Ver status
docker compose ps

# Parar (dados preservados)
docker compose down

# Logs de um servico
docker compose logs jupyter-1 --tail 50

# Resetar tudo
docker compose down
rm -rf data/
mkdir -p data/minio{1..4} data/dremio data/dremio-spill data/spark-events
chmod -R 777 data/dremio data/dremio-spill
docker compose up -d --build
```

### Fluxo de dados

```
JSON (upload) → MinIO (landing) → Spark (ETL) → MinIO (bronze/prata/ouro) → Dremio (SQL)
                                                                                  ↓
                                                                            Metabase / BI
```